# Exploração do dataset (n=100)

Ambiente de exploração livre sobre os artefatos processados. Nada aqui
reconsulta a API — tudo vem de `data/processed/` (e o bruto está em
`data/raw/`, por repositório).

Como abrir: `make lab` (ou `uv run jupyter lab`) na raiz do repositório.

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "config" / "metrics.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from govscore.figures import ARCH_COLOR, ARCH_LABEL  # paleta validada

scores = pd.read_csv(ROOT / "data/processed/scores.csv")
metrics = pd.read_parquet(ROOT / "data/processed/metrics.parquet")
ext = pd.read_csv(ROOT / "data/processed/external_indicators.csv")
df = scores.merge(ext[["repo", "dependents", "scorecard"]], on="repo")
df.head()

,repo,archetype,language,stars,score,subscore_artifacts,subscore_distribution,subscore_responsiveness,subscore_diversity,subscore_security,extracted_at,dependents,scorecard
0,torvalds/linux,federation,c,240237,56.894519,0.222222,1.000000,NaN,0.997337,0.0,2026-07-24,NaN,3.0
1,tensorflow/tensorflow,federation,c++,196481,73.949544,0.444444,0.915494,0.727047,0.694010,1.0,2026-07-23,NaN,7.3
2,rails/rails,federation,ruby,58636,77.080699,0.666667,1.000000,0.683333,0.849824,0.6,2026-07-23,NaN,6.8
3,freeCodeCamp/freeCodeCamp,federation,typescript,452563,81.355883,0.777778,1.000000,0.900000,0.760763,0.5,2026-07-23,NaN,6.4
4,react/react-native,federation,c++,126252,71.730919,0.666667,1.000000,0.566667,0.648728,0.6,2026-07-23,NaN,NaN


## Distribuição do score

In [2]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for arch, g in df.groupby("archetype"):
    ax.hist(g.score, bins=14, alpha=0.55, label=ARCH_LABEL[arch],
            color=ARCH_COLOR[arch])
ax.set_xlabel("score")
ax.legend()
plt.show()

/var/folders/vv/ckmfyvqd4hj331_36wmp26l80000gn/T/ipykernel_81116/4253423236.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Consultas rápidas

Exemplos — edite à vontade.

In [3]:
# Top 10 geral
df.nlargest(10, "score")[["repo", "archetype", "language", "score", "scorecard"]]

,repo,archetype,language,score,scorecard
17,nodejs/node,federation,javascript,94.590174,6.1
23,supabase/supabase,federation,typescript,92.001201,7.4
10,godotengine/godot,federation,c++,87.086102,5.4
52,vllm-project/semantic-router,club,go,85.701409,NaN
56,cilium/tetragon,club,c,85.593608,NaN
60,Expensify/App,club,typescript,85.444040,NaN
62,hashicorp/terraform-provider-azurerm,club,go,85.281140,5.4
9,rust-lang/rust,federation,rust,85.246542,7.1
74,ydb-platform/ydb,club,c++,83.763691,NaN
18,langgenius/dify,federation,typescript,83.383151,NaN


In [4]:
# Clubes em Rust ou Go, ordenados por diversidade
df.query("archetype == 'club' and language in ('rust', 'go')")\
  .sort_values("subscore_diversity", ascending=False)\
  [["repo", "language", "score", "subscore_diversity", "subscore_security"]]

,repo,language,score,subscore_diversity,subscore_security
62,hashicorp/terraform-provider-azurerm,go,85.281140,0.974953,1.000000
52,vllm-project/semantic-router,go,85.701409,0.902316,0.700000
70,asterinas/asterinas,rust,68.289652,0.841856,0.458333
57,vllm-project/aibrix,go,77.006532,0.669696,0.766667
67,mozilla/uniffi-rs,rust,54.581491,0.546672,0.000000
64,Gentleman-Programming/gentle-ai,go,39.140757,0.175834,0.600000


## Métricas cruas por arquétipo

In [5]:
CRUAS = ["responsiveness_median_first_response_hours",
         "responsiveness_pr_review_coverage",
         "distribution_elephant_factor",
         "distribution_contributor_retention",
         "security_releases_12m"]
metrics.groupby("archetype")[CRUAS].median().round(2)

,responsiveness_median_first_response_hours,responsiveness_pr_review_coverage,distribution_elephant_factor,distribution_contributor_retention,security_releases_12m
archetype,,,,,
club,27.31,0.92,3.0,0.39,14.0
federation,17.88,0.74,7.0,0.33,22.0
stadium,23.81,0.26,1.0,0.19,2.0
toy,22.09,0.00,1.0,0.75,3.0


In [6]:
# dispersão de uma métrica crua por arquétipo (troque a coluna à vontade)
COL = "responsiveness_pr_review_coverage"
fig, ax = plt.subplots(figsize=(8, 3))
for i, arch in enumerate(["federation", "stadium", "club", "toy"]):
    vals = metrics.loc[metrics.archetype == arch, COL].dropna()
    ax.scatter(vals, [i] * len(vals), s=18, alpha=0.7,
               color=ARCH_COLOR[arch])
ax.set_yticks(range(4),
              [ARCH_LABEL[a] for a in ["federation", "stadium", "club", "toy"]])
ax.set_xlabel(COL)
plt.show()

/var/folders/vv/ckmfyvqd4hj331_36wmp26l80000gn/T/ipykernel_81116/2376522078.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Mergulho em um repositório

In [7]:
recs = {r["repo"]: r for r in json.loads(
    (ROOT / "data/processed/full_metrics.json").read_text())["results"]}

def detalhe(repo):
    r = recs[repo]
    print(f"{repo} — score {r['score']:.1f} ({r['archetype']})")
    for k in ("subscores", "artifacts", "security", "responsiveness"):
        print(f"\n{k}:", json.dumps(r[k], indent=2, ensure_ascii=False))
    print(f"\nbruto em: data/raw/{repo.replace('/', '__')}/")

detalhe("nodejs/node")

nodejs/node — score 94.6 (federation)

subscores: {
  "artifacts": 0.8888888888888888,
  "distribution": 1.0,
  "responsiveness": 0.9647804418103448,
  "diversity": 0.8714895347932347,
  "security": 1.0
}

artifacts: {
  "readme": true,
  "contributing": true,
  "code_of_conduct": true,
  "license": true,
  "issue_template": false,
  "pull_request_template": true,
  "codeowners": true,
  "governance": true,
  "funding": true,
  "health_percentage": 100,
  "funding_inherited": true
}

security: {
  "security_policy": true,
  "ci_configured": true,
  "dependency_automation": true,
  "releases_12m": 30,
  "release_notes_share": 1.0
}

responsiveness: {
  "median_first_response_hours": 11.321388888888889,
  "pr_review_coverage": 1.0,
  "median_issue_close_hours": 30595.796388888888,
  "median_pr_merge_hours": 58.5025,
  "pr_merge_ratio": 0.72,
  "n_issues_sampled": 27,
  "n_prs_sampled": 50,
  "n_first_responses": 43
}

bruto em: data/raw/nodejs__node/


## Faltantes (nunca imputados)

In [8]:
faltantes = metrics.isna().mean().sort_values(ascending=False)
faltantes[faltantes > 0].round(2).head(12)

artifacts_funding_inherited                   0.95
security_security_policy_inherited            0.89
security_release_notes_share                  0.26
responsiveness_median_issue_close_hours       0.14
distribution_contributor_retention            0.12
responsiveness_median_first_response_hours    0.10
responsiveness_median_pr_merge_hours          0.10
responsiveness_pr_review_coverage             0.06
subscore_responsiveness                       0.04
responsiveness_pr_merge_ratio                 0.04
dtype: float64